# Example 11.3.1.3 / 1: Match a Function Name to What It Computes

Companion notebook for **11.3.1.3 GSL Probability Distributions** in the AMPL lab Documentation, section "Match a Function Name to What It Computes". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Match a Function Name to What It Computes

Every distribution uses the same name pattern. For a continuous distribution named `dist`:

| Name | Computes | Gaussian example | Result |
|---|---|---|---|
| `gsl_ran_dist` | a random variate | `gsl_ran_gaussian(1)` | `0.3064` (first draw) |
| `gsl_ran_dist_pdf` | the density $p(x)$ | `gsl_ran_gaussian_pdf(0.5, 1)` | `0.352065` |
| `gsl_cdf_dist_P` | the lower tail $P(x)=\Pr(X\le x)$ | `gsl_cdf_gaussian_P(0.5, 1)` | `0.691462` |
| `gsl_cdf_dist_Q` | the upper tail $Q(x)=1-P(x)$ | `gsl_cdf_gaussian_Q(0.5, 1)` | `0.308538` |
| `gsl_cdf_dist_Pinv` | the $x$ with $P(x)$ equal to the argument | `gsl_cdf_gaussian_Pinv(0.9, 1)` | `1.28155` |
| `gsl_cdf_dist_Qinv` | the $x$ with $Q(x)$ equal to the argument | `gsl_cdf_gaussian_Qinv(0.1, 1)` | `1.28155` |

A discrete distribution has the variate, the probability $p(k)$ and the two tails $P(k)$ and
$Q(k)=\Pr(X>k)$, but no inverses. Computing $Q$ directly keeps accuracy in a small upper tail,
where $1-P$ would round to 0. Each variate function must be declared `random`, as `gsl.ampl` does,
so that AMPL calls it again for each use instead of reusing one value.

A safety-stock calculation from the steel model: demand for coils has a forecast error with
standard deviation `sd_demand` tons, and production must cover it with probability `service`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
load amplgsl.dll;
function gsl_cdf_ugaussian_Pinv;  # standard normal quantile
function gsl_cdf_gaussian_Q;      # upper tail of a mean-0 normal

param sd_demand > 0;              # tons, standard deviation of forecast error
param service > 0, < 1;           # required probability of covering demand
param safety_stock = sd_demand * gsl_cdf_ugaussian_Pinv(service);  # tons
param shortfall_prob = gsl_cdf_gaussian_Q(safety_stock, sd_demand); # check: 1 - service

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.param["sd_demand"] = 120
ampl.param["service"] = 0.95

In [5]:
# run.py
print("safety stock:", round(ampl.param["safety_stock"].value(), 2))
print("shortfall probability:", round(ampl.param["shortfall_prob"].value(), 4))

safety stock: 197.38
shortfall probability: 0.05


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param sd_demand := 120;
param service := 0.95;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display safety_stock, shortfall_prob;

safety_stock = 197.382
shortfall_prob = 0.05

